In [26]:
import pandas as pd
# Load dataset and import pandas as pd
df = pd.read_csv("https://rossbulat.com/data/ab_nyc_2019.csv")

In [27]:
# Install required packages
!pip install torch torchvision torchaudio nltk pandas scikit-learn



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [28]:
# Import libraries
import nltk
nltk.download('vader_lexicon')
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/luistorres/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [29]:
# Check the number of nulls in each field
df.isnull().sum()
print(df.isnull().sum())


id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64


In [30]:
# Recheck the number of nulls in each field
df = df[pd.notnull(df['name'])]
df = df[pd.notnull(df['host_name'])]


In [31]:
# Recheck the number of nulls in each field
df.isnull().sum()
print(df.isnull().sum())


id                                    0
name                                  0
host_id                               0
host_name                             0
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10037
reviews_per_month                 10037
calculated_host_listings_count        0
availability_365                      0
dtype: int64


In [32]:
# Create combined text field
df = df[['name','host_name','neighbourhood','neighbourhood_group']].fillna("")
df['combined_text'] = df.apply(lambda x: ' '.join(x), axis=1)


In [33]:
# Creating a Lexicon-based sentiment labels
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
df['label'] = df['combined_text'].apply(lambda x: 1 if sia.polarity_scores(x)['compound'] > 0 else 0)


In [34]:
# Tokenization and vocabulary creation
def basic_tokenizer(text):
    return re.findall(r"\b\w+\b", text.lower())


In [35]:
# Build vocabulary for the model
all_tokens = []
for text in df['combined_text']:
    all_tokens.extend(basic_tokenizer(text))
vocab = {tok: idx+1 for idx, tok in enumerate(set(all_tokens))}
unk_index = 0  # reserve 0 for unknown
vocab_size = len(vocab) + 1


In [36]:
# Create the dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        tokens = basic_tokenizer(self.texts[idx])
        ids = torch.tensor([vocab.get(tok, unk_index) for tok in tokens], dtype=torch.long)
        return ids, torch.tensor(self.labels[idx], dtype=torch.float)


In [37]:
# Custom collate function i.e. overrides the default merge logic
def collate_batch(batch):
    texts, labels = zip(*batch)
    texts = nn.utils.rnn.pad_sequence(texts, batch_first=True, padding_value=unk_index)
    labels = torch.stack(labels)
    return texts, labels


In [38]:
# Split the dataset into Train/Test datasets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['combined_text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)
train_dataset = TextDataset(train_texts, train_labels)
test_dataset = TextDataset(test_texts, test_labels)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch)


In [39]:
# Building the Model
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SentimentModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=unk_index)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return out  # logits

model = SentimentModel(vocab_size, embed_dim=100, hidden_dim=128)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [40]:
# Training the loop
for epoch in range(5):
    epoch_loss = 0.0
    batch_count = 0
    model.train()
    for texts, labels in train_loader:
        outputs = model(texts).squeeze()
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        batch_count += 1
    avg_loss = epoch_loss / batch_count
    print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")


Epoch 1, Avg Loss: 0.3332
Epoch 2, Avg Loss: 0.1393
Epoch 3, Avg Loss: 0.0949
Epoch 4, Avg Loss: 0.0754
Epoch 5, Avg Loss: 0.0587


In [41]:
# Evaluating the model
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for texts, labels in test_loader:
        outputs = model(texts).squeeze()
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        y_true.extend(labels.tolist())
        y_pred.extend(preds.tolist())

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))


Accuracy: 0.9671510437986083
Precision: 0.949443413729128
Recall: 0.9754586609482964
F1: 0.9622752379833118


In [42]:
# Predict sentiment for new text
def predict_model(text):
    tokens = basic_tokenizer(text)
    ids = torch.tensor([vocab.get(tok, unk_index) for tok in tokens], dtype=torch.long).unsqueeze(0)
    output = torch.sigmoid(model(ids)).item()
    return "Positive" if output >= 0.5 else "Negative"

def predict_vader(text):
    score = sia.polarity_scores(text)['compound']
    return "Positive" if score > 0 else "Negative"

examples = [
    "Lovely apartment in Manhattan",
    "Terrible host, very rude",
    "Cozy room near Central Park",
    "The place was dirty and noisy",
    "Affordable stay in Brooklyn near the bridge",
    "Spacious upper floor in Brooklyn with great view"
]
for ex in examples:
    print(f"Text: {ex}")
    print(f"Model Prediction: {predict_model(ex)}")
    print(f"VADER Prediction: {predict_vader(ex)}")
    print("-"*40)


Text: Lovely apartment in Manhattan
Model Prediction: Positive
VADER Prediction: Positive
----------------------------------------
Text: Terrible host, very rude
Model Prediction: Negative
VADER Prediction: Negative
----------------------------------------
Text: Cozy room near Central Park
Model Prediction: Negative
VADER Prediction: Negative
----------------------------------------
Text: The place was dirty and noisy
Model Prediction: Positive
VADER Prediction: Negative
----------------------------------------
Text: Affordable stay in Brooklyn near the bridge
Model Prediction: Negative
VADER Prediction: Negative
----------------------------------------
Text: Spacious upper floor in Brooklyn with great view
Model Prediction: Positive
VADER Prediction: Positive
----------------------------------------


## Modern GPT-style Transformer Classifier

Create a GPT-style Transformer model for comparison. 


In [43]:
# GPT-style classifier: token + positional embeddings, causal self-attention, last-token pooling
class GPTClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, nhead=4, num_layers=2, dim_feedforward=256, max_len=32):
        super(GPTClassifier, self).__init__()
        self.max_len = max_len
        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=unk_index)
        self.position_embedding = nn.Embedding(max_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, dim_feedforward=dim_feedforward, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x):
        x = x[:, :self.max_len]
        batch_size, seq_len = x.shape

        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        h = self.token_embedding(x) + self.position_embedding(positions)

        pad_mask = (x == unk_index)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device) == float('-inf')

        h = self.transformer(h, mask=causal_mask, src_key_padding_mask=pad_mask)

        # Pool the hidden state at each sequence's last real (non-pad) token, GPT-2 classification style
        lengths = (~pad_mask).sum(dim=1).clamp(min=1) - 1
        last_hidden = h[torch.arange(batch_size), lengths]

        out = self.fc(last_hidden)
        return out  # logits

gpt_model = GPTClassifier(vocab_size, embed_dim=128, nhead=4, num_layers=2, dim_feedforward=256, max_len=32)
gpt_criterion = nn.BCEWithLogitsLoss()
gpt_optimizer = optim.Adam(gpt_model.parameters(), lr=0.001)


In [44]:
# Training the loop (GPT-style model)
for epoch in range(5):
    epoch_loss = 0.0
    batch_count = 0
    gpt_model.train()
    for texts, labels in train_loader:
        outputs = gpt_model(texts).squeeze()
        loss = gpt_criterion(outputs, labels)
        gpt_optimizer.zero_grad()
        loss.backward()
        gpt_optimizer.step()
        epoch_loss += loss.item()
        batch_count += 1
    avg_loss = epoch_loss / batch_count
    print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")


Epoch 1, Avg Loss: 0.2887
Epoch 2, Avg Loss: 0.1705
Epoch 3, Avg Loss: 0.1313
Epoch 4, Avg Loss: 0.1147
Epoch 5, Avg Loss: 0.1021


## Performance comparison

There is no improvement in performance compared to the classic Sentiment approach

In [45]:
# Evaluating the GPT-style model
gpt_model.eval()
gpt_y_true, gpt_y_pred = [], []
with torch.no_grad():
    for texts, labels in test_loader:
        outputs = gpt_model(texts).squeeze()
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        gpt_y_true.extend(labels.tolist())
        gpt_y_pred.extend(preds.tolist())

print("Accuracy:", accuracy_score(gpt_y_true, gpt_y_pred))
print("Precision:", precision_score(gpt_y_true, gpt_y_pred))
print("Recall:", recall_score(gpt_y_true, gpt_y_pred))
print("F1:", f1_score(gpt_y_true, gpt_y_pred))


Accuracy: 0.9572247237003684
Precision: 0.9233699305399955
Recall: 0.9818918274958304
F1: 0.9517321016166281


## Comparison of Sentiment analysis on new data

New model predicts new words exactly equal as the classic model 

In [46]:
# Predict sentiment for new text with the GPT-style model, compared against the LSTM and VADER
def predict_gpt_model(text):
    tokens = basic_tokenizer(text)
    ids = torch.tensor([vocab.get(tok, unk_index) for tok in tokens], dtype=torch.long).unsqueeze(0)
    output = torch.sigmoid(gpt_model(ids)).item()
    return "Positive" if output >= 0.5 else "Negative"

for ex in examples:
    print(f"Text: {ex}")
    print(f"LSTM Prediction: {predict_model(ex)}")
    print(f"GPT Prediction: {predict_gpt_model(ex)}")
    print(f"VADER Prediction: {predict_vader(ex)}")
    print("-"*40)

Text: Lovely apartment in Manhattan
LSTM Prediction: Positive
GPT Prediction: Positive
VADER Prediction: Positive
----------------------------------------
Text: Terrible host, very rude
LSTM Prediction: Negative
GPT Prediction: Negative
VADER Prediction: Negative
----------------------------------------
Text: Cozy room near Central Park
LSTM Prediction: Negative
GPT Prediction: Negative
VADER Prediction: Negative
----------------------------------------
Text: The place was dirty and noisy
LSTM Prediction: Positive
GPT Prediction: Negative
VADER Prediction: Negative
----------------------------------------
Text: Affordable stay in Brooklyn near the bridge
LSTM Prediction: Negative
GPT Prediction: Negative
VADER Prediction: Negative
----------------------------------------
Text: Spacious upper floor in Brooklyn with great view
LSTM Prediction: Positive
GPT Prediction: Positive
VADER Prediction: Positive
----------------------------------------


### comparison outcome
Both models were trained on the same VADER-derived weak labels, limiting the amount of additional information that a more expressive architecture could leverage. Given this constraint, the traditional sentiment analysis approach, despite not using a modern transformer architecture, was sufficient to achieve the task effectively. The results suggest that the added complexity of a transformer model may not provide significant benefits when the quality and richness of the training labels are limited.